# Specialiserede Modeller — 6: Autoencodere — kunsten at glemme det uvæsentlige

Dagens model er den mærkeligste, I har mødt: et netværk, der trænes til at
**genskabe sit eget input**. "Gæt hvad du lige har fået at se"?! Det lyder som verdens
nemmeste opgave — indtil man ser fidusen: midt i netværket sidder en **flaskehals** så
snæver, at billedet IKKE kan komme helskindet igennem. Netværket TVINGES til at
komprimere: gemme det væsentlige, smide resten væk.

Det giver tre superkræfter: **kompression**, **støjfjernelse** og
**anomali-detektion** — og som sidegevinst et smukt indblik i, hvad et netværk
egentlig "forstår". Data: jeres gamle venner, de håndskrevne MNIST-cifre (kendt stof,
så al opmærksomheden kan gå til den nye modeltype).

> Denne notebook er selvkørende og kræver kun viden fra Intro-ML — du kan tage emnets notebooks i den rækkefølge, du vil. Der er med vilje flere opgaver, end du kan nå: opgaver mærket **(ekstra)** er til de hurtige, og opgaver mærket **(find fejlen)** indeholder en bevidst fejl, som skal findes og rettes.

## Setup

In [ ]:
# Henter MNIST (nedskaleret) fra GitHub (Plan B: upload filerne via mappeikonet i Colab)
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/28-Data/MLData/mnist_traen_lille.csv.gz
!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/28-Data/MLData/mnist_test_lille.csv.gz

!wget -q -nc https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/98-Helpers/helpers.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from helpers import show_images, show_reconstructions

torch.manual_seed(42)
np.random.seed(42)

train_df = pd.read_csv("mnist_traen_lille.csv.gz").sample(n=8000, random_state=42).reset_index(drop=True)
test_df = pd.read_csv("mnist_test_lille.csv.gz").reset_index(drop=True)
print("træning:", train_df.shape, "| test:", test_df.shape)

> **Plan B:** Fejler Kaggle, ligger der mindre udgaver i Intro-ML-mappen på GitHub —
> fjern `#`'erne (og spring `sample` over; filerne er allerede skåret ned):

In [ ]:
#  Plan B — kør KUN hvis Kaggle-cellen fejlede:
# traen_df = pd.read_csv("https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/28-Data/MLData/mnist_traen_lille.csv.gz")
# test_df = pd.read_csv("https://raw.githubusercontent.com/UNF-Science-Camps/KIC26/main/28-Data/MLData/mnist_test_lille.csv.gz")
# print(traen_df.shape, test_df.shape)

In [ ]:
X_train = torch.tensor(train_df.drop(columns=["label"]).values / 255.0, dtype=torch.float32)
X_test = torch.tensor(test_df.drop(columns=["label"]).values / 255.0, dtype=torch.float32)
y_train = torch.tensor(train_df["label"].values, dtype=torch.long)   # labels bruges KUN til at farve plots!
y_test = torch.tensor(test_df["label"].values, dtype=torch.long)

print(X_train.shape, "— flade billeder som i Intro-ML")

# 1: Flaskehalsen — 784 tal ind, 32 tal i midten, 784 tal ud

En autoencoder består af to halvdele:

- **Encoderen** presser billedet ned: 784 pixels →... → fx **32 tal** (flaskehalsen —
 også kaldet den *latente kode*)
- **Decoderen** puster det op igen: 32 tal →... → 784 pixels

Og tabsfunktionen? Bare MSE mellem input og output: "hvor tæt kom du på originalen?".
Bemærk det geniale: **der bruges INGEN labels** — billedet er sit eget facit. Det er
unsupervised learning, ligesom clustering, bare med neurale netværk.

Hvorfor lærer flaskehalsen noget klogt? Fordi 32 tal er ALT for lidt til 784 pixels —
netværket kan ikke gemme pixel-værdierne råt. Det tvinges til at opdage, at håndskrevne
cifre slet ikke er vilkårlige pixelmønstre, men varianter over ~10 former med
buer, streger og hældninger — og DET kan beskrives med få tal:

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, bottleneck=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 128), nn.ReLU(),
            nn.Linear(128, bottleneck))            # ned til fx 32 tal
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, 128), nn.ReLU(),
            nn.Linear(128, 784), nn.Sigmoid())     # sigmoid: pixels skal ligge i [0, 1]

    def forward(self, x):
        kode = self.encoder(x)
        return self.decoder(kode)

model = Autoencoder(bottleneck=32)
print(model)

In [ ]:
torch.manual_seed(42)
model = Autoencoder(bottleneck=32)
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    for i in range(0, len(X_train), 64):
        X_batch = X_train[i:i + 64]
        optimizer.zero_grad()
        loss = loss_fn(model(X_batch), X_batch)    # facit = inputtet selv!
        loss.backward()
        optimizer.step()
    print(f"epoke {epoch + 1}: tab {loss.item():.4f}")

In [ ]:
with torch.no_grad():
    reconstructions = model(X_test[:8])

show_reconstructions(X_test[:8], reconstructions, n=8,
                     title="784 pixels → 32 tal → 784 pixels")

Billederne har været igennem et nåleøje på 32 tal — en **kompression på 96 %** — og
er stadig tydeligt læsbare (lidt blødere i kanterne, detaljer er ofret). Netværket har
selv opfundet et "ciffer-sprog" med 32 ord.

## Hvor lille kan flaskehalsen blive?

Lad os presse citronen: samme netværk, fire flaskehals-størrelser:

In [ ]:
for bottleneck in [2, 8, 32, 64]:
    torch.manual_seed(42)
    m = Autoencoder(bottleneck=bottleneck)
    optimizer = torch.optim.Adam(m.parameters(), lr=0.001)
    for epoch in range(10):
        for i in range(0, len(X_train), 64):
            X_batch = X_train[i:i + 64]
            optimizer.zero_grad()
            loss = loss_fn(m(X_batch), X_batch)
            loss.backward()
            optimizer.step()
    with torch.no_grad():
        rek = m(X_test[:8])
    show_reconstructions(X_test[:8], rek, n=8, title=f"flaskehals = {bottleneck} tal")

Ved 64 og 32: fine cifre. Ved 8: læsbart men udglattet. Ved **2**: netværket kan
kun huske "cirka hvilket ciffer" — alt andet er væk. Og netop 2-tals-udgaven gemmer på
notebookens flotteste figur...

## Det latente rum: netværkets mentale landkort

Med flaskehals = 2 kan hvert billede beskrives med to tal — altså et PUNKT i planen!
Lad os encode hele testsættet og farve punkterne efter, hvilket ciffer de er. Husk:
netværket har ALDRIG set labels:

In [ ]:
torch.manual_seed(42)
model2d = Autoencoder(bottleneck=2)
optimizer = torch.optim.Adam(model2d.parameters(), lr=0.001)
for epoch in range(10):
    for i in range(0, len(X_train), 64):
        X_batch = X_train[i:i + 64]
        optimizer.zero_grad()
        loss = loss_fn(model2d(X_batch), X_batch)
        loss.backward()
        optimizer.step()

with torch.no_grad():
    koder = model2d.encoder(X_test)

plt.figure(figsize=(8, 7))
scatter = plt.scatter(koder[:, 0], koder[:, 1], c=y_test, cmap="tab10", s=10, alpha=0.7)
plt.colorbar(scatter, label="ciffer")
plt.title("Hele testsættet som punkter i det latente rum")
plt.show()

**Se!** Uden at kende ét eneste label har netværket sorteret cifrene i egne områder —
0'erne bor ét sted, 1'erne et andet... Kompression TVANG det til at forstå strukturen.
(Områderne overlapper hist og her — med kun 2 tal er der trængsel. Men at strukturen
overhovedet opstår af sig selv, er en af de smukkeste observationer i deep learning.)

### Opgaver

##### Opgave 1.1
Forklar med jeres egne ord: hvorfor lærer netværket ikke bare at "kopiere billedet
igennem"? Hvad er det præcis, flaskehalsen forhindrer — og hvad tvinger den i stedet
netværket til at gøre?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

##### Opgave 1.2
Udfyld tabslinjen i træningsloopet — hvad skal rekonstruktionen sammenlignes med?

In [ ]:
torch.manual_seed(42)
m_test = Autoencoder(bottleneck=16)
optimizer = torch.optim.Adam(m_test.parameters(), lr=0.001)
for epoch in range(3):
    for i in range(0, len(X_train), 64):
        X_batch = X_train[i:i + 64]
        optimizer.zero_grad()
        loss = loss_fn(m_test(X_batch), ...)
        loss.backward()
        optimizer.step()
print(f"tab: {loss.item():.4f}")

##### Opgave 1.3
Kig på de fire rekonstruktions-figurer (flaskehals 2/8/32/64). Beskriv præcis, HVAD
der forsvinder, når flaskehalsen strammes — er det tilfældige pixels, eller er det
bestemte TYPER af information? Og ved flaskehals 2: hvad gætter netværket åbenlyst
ud fra?

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

##### Opgave 1.4
Skil maskinen ad: encode ét billede og PRINT dets 32-tals kode — og decode den så
igen. Prøv at ændre ét af de 32 tal kraftigt (fx sæt `kode[0, 5] = 10.0`) før
decodningen. Hvad sker der med billedet?

In [ ]:
image = X_test[0:1]

with torch.no_grad():
    kode = model.encoder(image)
    print("billedet som 32 tal:")
    print(kode.numpy().round(2))

    # ← pil ved koden her, fx: kode[0, 5] = 10.0
    reconstructed = model.decoder(kode)

show_reconstructions(image, reconstructed, n=1)

##### Opgave 1.5 (find fejlen)
En kammerats autoencoder rekonstruerer nogle underligt "beskidte" billeder med grå
baggrund, og tabet sidder fast højere end jeres. Sammenlign kammeratens decoder med
den rigtige — hvad mangler der til sidst, og hvorfor betyder det noget, når pixels
ligger mellem 0 og 1?

In [ ]:
class BuddyAutoencoder(nn.Module):
    def __init__(self, bottleneck=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 128), nn.ReLU(),
            nn.Linear(128, bottleneck))
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, 128), nn.ReLU(),
            nn.Linear(128, 784))                  # hmm...

    def forward(self, x):
        return self.decoder(self.encoder(x))

torch.manual_seed(42)
m_k = BuddyAutoencoder()
optimizer = torch.optim.Adam(m_k.parameters(), lr=0.001)
for epoch in range(5):
    for i in range(0, len(X_train), 64):
        X_batch = X_train[i:i + 64]
        optimizer.zero_grad()
        loss = loss_fn(m_k(X_batch), X_batch)
        loss.backward()
        optimizer.step()
print(f"tab: {loss.item():.4f}")
with torch.no_grad():
    show_reconstructions(X_test[:8], m_k(X_test[:8]), n=8)

##### Opgave 1.6
Zoom ind i det latente rum: hvilke cifre overlapper mest i 2D-plottet? Tegn plottet
igen med KUN to udvalgte cifre (fx 4 og 9), og find et par, der er (næsten) perfekt
adskilt. Giver forvekslingerne mening rent visuelt?

In [ ]:
a, b = 4, 9      # ← prøv også (0, 1), (3, 8), (1, 7) ...
selection = (y_test == a) | (y_test == b)

plt.figure(figsize=(7, 6))
plt.scatter(koder[selection, 0], koder[selection, 1], c=y_test[selection], cmap="tab10", s=12)
plt.title(f"latent rum: kun {a} og {b}")
plt.show()

##### Opgave 1.7 (ekstra)
**Morphing-maskinen:** tag to billeder (fx et 0 og et 1), encode dem til to punkter i
det latente rum — og decode så 10 punkter på LINJEN imellem dem. Udfyld
blandings-linjen og se det ene ciffer glide over i det andet.

In [ ]:
image_a = X_test[y_test == 0][0:1]
image_b = X_test[y_test == 1][0:1]

with torch.no_grad():
    kode_a = model2d.encoder(image_a)
    kode_b = model2d.encoder(image_b)

    interpolations = []
    for t in np.linspace(0, 1, 10):
        mixed_kode = ...                      # (1-t) dele kode_a + t dele kode_b
        interpolations.append(model2d.decoder(mixed_kode))

fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for axis, image in zip(axes, interpolations):
    axis.imshow(image.reshape(28, 28), cmap="gray")
    axis.axis("off")
plt.show()

##### Opgave 1.8
784 tal → 32 tal er ~96 % kompression, og cifrene overlever. Hvorfor kan denne
autoencoder så IKKE bare erstatte JPEG/zip til alle billeder? (Prøv evt. tanken: hvad
ville der ske, hvis I fodrede den med et foto af jeres kat?)

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*

# 2: Støjfjernelse og alarmklokker

## Denoising: giv netværket et sværere job

Nu bliver det smart for alvor: giv netværket **støjede billeder som input**, men de
**RENE billeder som facit**. Så lærer det ikke bare at komprimere — det lærer at
**fjerne støj**. Og det kan det, fordi det ved, hvordan cifre SKAL se ud (støj passer
ikke ind i ciffer-sproget og bliver simpelthen ikke gen-tegnet):

In [ ]:
def add_noise(X, strength=0.3):
    noisy = X + strength * torch.randn(X.shape)
    return noisy.clamp(0, 1)              # pixels skal blive i [0, 1]

torch.manual_seed(42)
denoiser = Autoencoder(bottleneck=32)
optimizer = torch.optim.Adam(denoiser.parameters(), lr=0.001)

for epoch in range(10):
    for i in range(0, len(X_train), 64):
        X_clean = X_train[i:i + 64]
        X_noisy = add_noise(X_clean)
        optimizer.zero_grad()
        loss = loss_fn(denoiser(X_noisy), X_clean)    # støjet ind → RENT facit!
        loss.backward()
        optimizer.step()
    print(f"epoke {epoch + 1}: tab {loss.item():.4f}")

In [ ]:
torch.manual_seed(7)
X_test_noisy = add_noise(X_test[:8])

with torch.no_grad():
    cleaned = denoiser(X_test_noisy)

show_reconstructions(X_test_noisy, cleaned, n=8,
                     title="øverst: hvad netværket FÅR — nederst: hvad det leverer")

Fra sne-storm til læsbare cifre. Præcis samme princip bruges i virkeligheden til at
rense skanninger, gamle fotos og astronomi-billeder — og tankegangen "lær at fjerne
støj" er tilmed kernen i diffusions-modeller (dem bag AI-billedgeneratorer!).

## Anomali-detektion: alarmen der ikke skal have eksempler

Sidste superkraft. Autoencoderen er EKSPERT i cifre — og netop derfor **elendig** til
alt andet. Fodrer man den med noget, der ikke ligner træningsdataene, bliver
rekonstruktionen dårlig, og **rekonstruktionsfejlen eksploderer**. Det kan bruges som
alarm: høj fejl = "det her har jeg aldrig set før!"

Det geniale: man behøver INGEN eksempler på det unormale for at træne alarmen — kun
masser af normalt. (Tænk kreditkort: millioner af normale køb, næsten ingen kendte
svindler.) Vi tester med "unormale" billeder, vi nemt kan lave: inverterede cifre
(hvid baggrund, sort skrift) og ren støj:

In [ ]:
def reconstruction_error(model, X):
    with torch.no_grad():
        rek = model(X)
    return ((rek - X) ** 2).mean(dim=1)          # én fejl PR. BILLEDE

normal = X_test[:500]
inverted = 1.0 - X_test[500:1000]             # negativ-udgaver
torch.manual_seed(1)
noise_images = torch.rand(500, 784)             # ren tilfældighed

error_normal = reconstruction_error(model, normal)
error_inverted = reconstruction_error(model, inverted)
error_noise = reconstruction_error(model, noise_images)

plt.hist(error_normal, bins=40, alpha=0.6, label="normale cifre")
plt.hist(error_inverted, bins=40, alpha=0.6, label="inverterede cifre")
plt.hist(error_noise, bins=40, alpha=0.6, label="ren støj")
plt.xlabel("rekonstruktionsfejl")
plt.ylabel("antal billeder")
plt.legend()
plt.title("Alarmen: unormale input får høj fejl")
plt.show()

Tre pænt adskilte pukler! Sæt en grænse mellem dem, og I har en vagthund, der aldrig
har set en indbrudstyv, men ved præcis, hvordan "normalt" ser ud.

### Opgaver

##### Opgave 2.1
Skru på støjstyrken: test denoiseren (trænet på styrke 0.3) med støj på **0.1, 0.5 og
1.0**. Hvornår giver den op — og HVORDAN giver den op? (Gætter den forkert, eller
tegner den bare grød?)

In [ ]:
strength = 0.1      # ← prøv 0.1, 0.5, 1.0
torch.manual_seed(7)
X_s = add_noise(X_test[:8], strength=strength)
with torch.no_grad():
    cleaned = denoiser(X_s)
show_reconstructions(X_s, cleaned, n=8, title=f"støjstyrke {strength}")

##### Opgave 2.2
Udfyld støj-funktionen: læg gaussisk støj til og "clamp" resultatet, så pixelværdierne
bliver i [0, 1].

In [ ]:
def my_noise(X, strength=0.3):
    noisy = X + ...
    return noisy....(0, 1)

show_images(my_noise(X_test[:5]), y_test[:5], n=5)

##### Opgave 2.3
Sammenlign den ALMINDELIGE autoencoder (`model`) med `denoiser`-modellen på de samme
støjede billeder. Hvem renser bedst — og hvorfor giver det mening, når de har set
forskellige ting under træningen?

In [ ]:
torch.manual_seed(7)
X_s = add_noise(X_test[:8])

with torch.no_grad():
    show_reconstructions(X_s, model(X_s), n=8, title="almindelig autoencoder")
    show_reconstructions(X_s, denoiser(X_s), n=8, title="denoiser")

##### Opgave 2.4 (find fejlen)
Denne denoiser-træning ser rigtig ud... men modellen lærer bare at beholde støjen. Kig
GODT på tabslinjen: hvad sammenlignes rekonstruktionen med — og hvad SKULLE den
sammenlignes med?

In [ ]:
torch.manual_seed(42)
d_error = Autoencoder(bottleneck=32)
optimizer = torch.optim.Adam(d_error.parameters(), lr=0.001)
for epoch in range(5):
    for i in range(0, len(X_train), 64):
        X_clean = X_train[i:i + 64]
        X_noisy = add_noise(X_clean)
        optimizer.zero_grad()
        loss = loss_fn(d_error(X_noisy), X_noisy)
        loss.backward()
        optimizer.step()

torch.manual_seed(7)
X_s = add_noise(X_test[:8])
with torch.no_grad():
    show_reconstructions(X_s, d_error(X_s), n=8, title="hmm — støjen er der stadig?")

##### Opgave 2.5
Udfyld fejl-pr.-billede-beregningen (kvadrér forskellen og tag gennemsnittet — men
KUN over pixel-dimensionen, så hvert billede får sin egen fejl).

In [ ]:
def my_reconstruction_error(model, X):
    with torch.no_grad():
        rek = model(X)
    return ((rek - X) ** 2).mean(...)

error = my_reconstruction_error(model, X_test[:10])
print(error)              # 10 tal — ét pr. billede

##### Opgave 2.6
Byg selv alarmen færdig: vælg en alarm-grænse ud fra histogrammet, og beregn (a) hvor
stor en andel af de NORMALE cifre der fejlagtigt udløser alarm, og (b) hvor stor en
andel af de INVERTEREDE der bliver fanget. Flyt grænsen og se de to tal trække i hver
sin retning.

In [ ]:
threshold_boundary = 0.03     # ← aflæs et fornuftigt sted i histogrammet, og prøv flere

false_alarms = (error_normal > threshold_boundary).float().mean()
caught = (error_inverted > threshold_boundary).float().mean()
print(f"grænse {threshold_boundary}: falske alarmer {false_alarms.item():.1%}, fangede {caught.item():.1%}")

##### Opgave 2.7 (ekstra)
Hvilket CIFFER er sværest at rekonstruere? Beregn den gennemsnitlige
rekonstruktionsfejl pr. ciffer-klasse (for-løkke over 0-9) og plot som søjler. Har I
en teori om, hvorfor netop dét ciffer er svært?

In [ ]:
error_alle = reconstruction_error(model, X_test)

mean_pr_digit = []
for digit in range(10):
    mean_pr_digit.append(...)

plt.bar(range(10), mean_pr_digit)
plt.xlabel("ciffer")
plt.ylabel("gns. rekonstruktionsfejl")
plt.xticks(range(10))
plt.show()

##### Opgave 2.8
Nævn to virkelige systemer, hvor "træn kun på det normale, og slå alarm ved høj
rekonstruktionsfejl" ville være oplagt — og beskriv for mindst ét af dem, hvad
metodens svaghed ville være i praksis. (Hint til svagheden: hvad sker der, når
"normalt" ændrer sig med tiden?)

*(Tænk over det, og diskutér med din sidemand — I skal ikke skrive svaret ned.)*